# Trying online download

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import datetime
from datasets import load_dataset
from tqdm import tqdm
import torchvision.transforms as transforms
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Compute Device: {device.upper()}")

✅ Compute Device: CUDA


In [2]:
from huggingface_hub import login

# It will prompt you for your token right below the cell!
login()

In [ ]:
import datasets
import logging

# 1. Force HF to show the progress bar even in WSL
datasets.logging.enable_progress_bar()

# 2. Force HF to print every internal network request it makes
datasets.logging.set_verbosity_info()

print("🚀 Attempting transparent download...")
dataset = datasets.load_dataset("princeton-vl/LayeredDepth", split="validation")
print("✅ Done!")

🚀 Attempting transparent download...


KeyboardInterrupt: 

In [3]:
# Robust Download Function to fix [Errno 104] Connection Reset
def download_with_retry(repo_id, split_name, max_retries=5):
    for attempt in range(max_retries):
        try:
            print(f"⏳ Attempt {attempt + 1}/{max_retries}: Connecting to Hugging Face...")
            ds = load_dataset(repo_id, split=split_name)
            print("✅ Download Successful!")
            return ds
        except Exception as e:
            print(f"❌ Network drop detected: {e}")
            if attempt < max_retries - 1:
                wait_time = 5 * (attempt + 1)
                print(f"🔄 Retrying in {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                raise RuntimeError("Failed to download dataset after multiple attempts. Check your VPN/Firewall.")

dataset = download_with_retry("princeton-vl/LayeredDepth", "validation")

⏳ Attempt 1/5: Connecting to Hugging Face...


KeyboardInterrupt: 

# Directly downloading and using

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import datetime
from datasets import load_dataset
from tqdm import tqdm
import torchvision.transforms as transforms

# Verify GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Compute Device: {device.upper()}")

print("⏳ Loading local offline Parquet file...")
# We tell HF to load our local file. 
# Note: HF automatically assigns local files to a "train" split internally, even if it's validation data.
dataset = load_dataset("parquet", data_files="0000.parquet", split="train")

print(f"✅ Successfully loaded {len(dataset)} images instantly from local disk!")



✅ Compute Device: CUDA
⏳ Loading local offline Parquet file...
✅ Successfully loaded 68 images instantly from local disk!


In [2]:
# 1. Grab the correct annotation key
sample = dataset[0]
annotation_key = 'tuples.json' if 'tuples.json' in sample else list(sample.keys())[-1] 

# 2. Extract the 'layer_all' test data
layer_data = sample[annotation_key]['layer_all']

print("--- The Final Data Structure ---")
print(f"Type: {type(layer_data)}")

# 3. Print the exact column names Princeton used
if isinstance(layer_data, dict):
    print(f"Column Names: {list(layer_data.keys())}")
    
    # Safely print the very first value of each column
    print("\n--- First Quadruplet Point ---")
    for col_name, col_array in layer_data.items():
        if isinstance(col_array, list) and len(col_array) > 0:
            print(f"{col_name}: {col_array[0]}")
        else:
            print(f"{col_name}: {col_array}")
else:
    print("It's a list! Here is the first item:")
    print(layer_data[0])

--- The Final Data Structure ---
Type: <class 'dict'>
Column Names: ['pairs', 'trips', 'quads']

--- First Quadruplet Point ---
pairs: {'p1': [953, 3424, 3], 'p2': [954, 3419, 3], 'is_real': True}
trips: {'p1': [953, 3424, 3], 'p2': [954, 3419, 3], 'p3': [954, 3414, 3], 'is_real': True}
quads: {'p1': [953, 3424, 3], 'p2': [954, 3419, 3], 'p3': [954, 3414, 3], 'p4': [955, 3409, 3], 'is_real': True}


In [3]:
class LayeredDepthReplication(nn.Module):
    def __init__(self, strategy="multi_head"):
        """
        strategy options: "multi_head", "recurrent", "concat"
        """
        super().__init__()
        self.strategy = strategy
        print(f"⚙️ Initializing Architecture Strategy: {strategy.upper()}")
        
        # Base Encoder (DINOv2 - The engine of Depth Anything)
        self.encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        for param in self.encoder.parameters():
            param.requires_grad = False # Freeze for testing

        # A standard lightweight decoder block
        def build_decoder(in_channels=384):
            return nn.Sequential(
                nn.Conv2d(in_channels, 128, 3, padding=1), nn.ReLU(),
                nn.Upsample(scale_factor=2, mode='bilinear'),
                nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),
                nn.Upsample(scale_factor=7, mode='bilinear'), # Up to 224x224
                nn.Conv2d(64, 1, 3, padding=1), nn.ReLU()
            )

        # Build architecture based on strategy
        if strategy == "multi_head":
            self.head1 = build_decoder(384) # Predicts Layer 1 (Glass)
            self.head2 = build_decoder(384) # Predicts Layer 2 (Background)
            
        elif strategy == "recurrent":
            self.base_decoder = build_decoder(384)
            # Context processor to merge Layer 1 depth back into features
            self.context_proj = nn.Conv2d(1, 384, kernel_size=14, stride=14) 
            self.layer2_decoder = build_decoder(384)
            
        elif strategy == "concat":
            # We add an extra channel for the "Layer Index" prompt
            self.prompt_proj = nn.Linear(1, 384)
            self.decoder = build_decoder(384)

    def extract_features(self, x):
        features = self.encoder.forward_features(x)["x_norm_patchtokens"]
        B, N, C = features.shape
        H = W = int(np.sqrt(N))
        return features.permute(0, 2, 1).view(B, C, H, W)

    def forward(self, x, target_layer=1):
        feats = self.extract_features(x)
        B, C, H, W = feats.shape

        if self.strategy == "multi_head":
            return self.head1(feats) if target_layer == 1 else self.head2(feats)

        elif self.strategy == "recurrent":
            depth1 = self.base_decoder(feats)
            if target_layer == 1: return depth1
            # Recurrent loop: Inject depth1 back into features
            context = self.context_proj(depth1)
            return self.layer2_decoder(feats + context)

        elif self.strategy == "concat":
            # Inject prompt (1.0 for layer 1, 2.0 for layer 2)
            prompt = torch.tensor([[float(target_layer)]]).to(x.device)
            prompt_vec = self.prompt_proj(prompt).view(B, C, 1, 1).expand(-1, -1, H, W)
            return self.decoder(feats + prompt_vec)

# Instantiate the model (You can change "multi_head" to "recurrent" or "concat")
model = LayeredDepthReplication(strategy="concat").to(device)

⚙️ Initializing Architecture Strategy: CONCAT
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /home/yash_gupta/.cache/torch/hub/main.zip


/home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /home/yash_gupta/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:20<00:00, 4.27MB/s]


In [4]:
def evaluate_image_tuples(predicted_depth_map, ground_truth_sample, original_size):
    correct = 0
    total = 0
    
    # Safely navigate the nested dictionary we discovered!
    try:
        annotation_key = 'tuples.json' if 'tuples.json' in ground_truth_sample else list(ground_truth_sample.keys())[-1]
        pairs_list = ground_truth_sample[annotation_key]['layer_all']['pairs']
    except KeyError:
        return 0, 0 # Skip if the image is missing this specific test data
        
    orig_w, orig_h = original_size
    pred_h, pred_w = predicted_depth_map.shape[2:]
    
    # We must scale the 4K coordinates down to our 224x224 tensor
    scale_x = pred_w / orig_w
    scale_y = pred_h / orig_h

    depth_array = predicted_depth_map[0, 0].cpu().numpy()

    for pair in pairs_list:
        # Only evaluate valid real-world points
        if not pair.get('is_real', True):
            continue
            
        p1, p2 = pair['p1'], pair['p2']
        x1, y1, rank1 = p1[0], p1[1], p1[2]
        x2, y2, rank2 = p2[0], p2[1], p2[2]
        
        # If they are on the same depth layer, we skip them for strict ordinal evaluation
        if rank1 == rank2:
            continue
            
        # 1 means p1 is closer (smaller rank). -1 means p2 is closer.
        gt_rel = 1 if rank1 < rank2 else -1
        
        # Scale and clamp coordinates
        px1 = min(max(int(x1 * scale_x), 0), pred_w - 1)
        py1 = min(max(int(y1 * scale_y), 0), pred_h - 1)
        px2 = min(max(int(x2 * scale_x), 0), pred_w - 1)
        py2 = min(max(int(y2 * scale_y), 0), pred_h - 1)
        
        # Look up the predicted depth from our neural network
        d1 = depth_array[py1, px1]
        d2 = depth_array[py2, px2]
        
        # NOTE: Standard models predict smaller values = closer. 
        # If using pre-trained Depth Anything V2 directly, you may need to flip this to '>' !
        pred_rel = 1 if d1 < d2 else -1
        
        if pred_rel == gt_rel:
            correct += 1
        total += 1
        
    return correct, total

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

total_correct = 0
total_pairs = 0

print(f"🚀 Running Evaluation on Local Dataset...")

model.eval()
with torch.no_grad():
    # Test on the first 10 images to verify the math
    for i in range(len(dataset)): 
        sample = dataset[i]
        img_pil = sample['image.png'].convert("RGB")
        img_tensor = transform(img_pil).unsqueeze(0).to(device)
        
        # Forward pass through the model
        pred_depth = model(img_tensor, target_layer=1)
        
        # Evaluate using our new dictionary-aware function
        correct, total = evaluate_image_tuples(pred_depth, sample, img_pil.size)
        
        total_correct += correct
        total_pairs += total
        
        acc = (correct / total) * 100 if total > 0 else 0
        # print(f"Image {i+1} | Analyzed {total} valid pairs | Accuracy: {acc:.1f}%")

final_acc = (total_correct / total_pairs) * 100 if total_pairs > 0 else 0
print("\n" + "="*50)
print(f"✅ Baseline Validation Complete!")
print(f"Total Valid Point Pairs Evaluated: {total_pairs}")
print(f"Final Architecture Accuracy: {final_acc:.2f}%")
print("="*50)

🚀 Running Evaluation on Local Dataset...

✅ Baseline Validation Complete!
Total Valid Point Pairs Evaluated: 10770
Final Architecture Accuracy: 37.48%


# continue with the training to replicate the results

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import os
from datasets import load_dataset
from tqdm import tqdm
import torchvision.transforms as transforms
from huggingface_hub import login

# 1. Login (Run this once or ensure you have HF_TOKEN set)
# login() 

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Compute Device: {device.upper()}")

# 2. Download Synthetic Dataset (Training) and Real Dataset (Validation)
def get_data():
    print("⏳ Downloading Synthetic Training Chunk (LayeredDepth-Syn)...")
    # We take the first 1,000 samples for a quick verification run
    train_ds = load_dataset("princeton-vl/LayeredDepth-Syn", split="train", streaming=True).take(1000)
    
    print("⏳ Downloading Real-World Validation (LayeredDepth)...")
    val_ds = load_dataset("princeton-vl/LayeredDepth", split="validation")
    
    return train_ds, val_ds

train_dataset, val_dataset = get_data()

✅ Compute Device: CUDA
⏳ Downloading Synthetic Training Chunk (LayeredDepth-Syn)...


Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

⏳ Downloading Real-World Validation (LayeredDepth)...


Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

...

data/test-00028-of-00031.parquet:   0%|          | 0.00/536M [00:00<?, ?B/s]

data/test-00029-of-00031.parquet:   0%|          | 0.00/681M [00:00<?, ?B/s]

data/test-00030-of-00031.parquet:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/300 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1200 [00:00<?, ? examples/s]

In [16]:
class SILogLoss(nn.Module):
    def __init__(self, lambd=0.5):
        super().__init__()
        self.lambd = lambd

    def forward(self, pred, target):
        # Ensure target and pred are the same size for bitmasking
        if pred.shape != target.shape:
            pred = F.interpolate(pred, size=target.shape[-2:], mode='bilinear', align_corners=False)
            
        valid_mask = (target > 0).detach()
        if valid_mask.sum() == 0:
            return torch.tensor(0.0, requires_grad=True).to(pred.device)

        # Log differences
        diff_log = torch.log(pred[valid_mask] + 1e-6) - torch.log(target[valid_mask] + 1e-6)
        
        # SILog Formula
        loss = torch.sqrt(torch.pow(diff_log, 2).mean() - self.lambd * torch.pow(diff_log.mean(), 2))
        return loss

In [17]:
class LayeredDepthModel(nn.Module):
    def __init__(self, strategy="concat"):
        super().__init__()
        self.strategy = strategy
        # Load DINOv2
        self.encoder = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
        for param in self.encoder.parameters():
            param.requires_grad = False # Keep backbone frozen as per Paper's initial experiments

        def build_decoder(in_channels=384):
            return nn.Sequential(
                nn.Conv2d(in_channels, 128, 3, padding=1), nn.ReLU(),
                nn.Upsample(scale_factor=2, mode='bilinear'),
                nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(),
                nn.Upsample(scale_factor=7, mode='bilinear'),
                nn.Conv2d(64, 1, 3, padding=1), nn.Softplus() # Softplus ensures depth > 0
            )

        if strategy == "multi_head":
            self.decoder1 = build_decoder(384)
            self.decoder2 = build_decoder(384)
        elif strategy == "recurrent":
            self.base_decoder = build_decoder(384)
            self.context_proj = nn.Conv2d(1, 384, 1)
            self.layer2_decoder = build_decoder(384)
        elif strategy == "concat":
            self.prompt_proj = nn.Linear(1, 384)
            self.decoder = build_decoder(384)

    def forward(self, x, target_layer=1):
        features = self.encoder.forward_features(x)["x_norm_patchtokens"]
        B, N, C = features.shape
        H = W = int(np.sqrt(N))
        feats = features.permute(0, 2, 1).view(B, C, H, W)

        if self.strategy == "multi_head":
            return self.decoder1(feats) if target_layer == 1 else self.decoder2(feats)
        elif self.strategy == "recurrent":
            d1 = self.base_decoder(feats)
            if target_layer == 1: return d1
            # Resize d1 to match patch grid [16x16]
            d1_low = F.interpolate(d1, size=(H, W), mode='bilinear')
            context = self.context_proj(d1_low)
            return self.layer2_decoder(feats + context)
        elif self.strategy == "concat":
            prompt = torch.tensor([[float(target_layer)]]).to(x.device)
            p_vec = self.prompt_proj(prompt).view(B, C, 1, 1).expand(-1, -1, H, W)
            return self.decoder(feats + p_vec)

In [41]:
def evaluate_model(model, val_ds, limit=50):
    model.eval()
    total_correct = 0
    total_pairs = 0
    
    # We use the same dictionary-aware logic we reverse-engineered
    with torch.no_grad():
        for i in range(limit):
            sample = val_ds[i]
            img_pil = sample['image.png'].convert("RGB")
            img_t = transform_img(img_pil).unsqueeze(0).to(device)
            
            # Predict Layer 1
            pred_depth = model(img_t, target_layer=1)
            
            # Use our previously defined evaluate_image_tuples logic
            correct, total = evaluate_image_tuples(pred_depth, sample, img_pil.size)
            total_correct += correct
            total_pairs += total
            
    return (total_correct / total_pairs) * 100 if total_pairs > 0 else 0

def run_experiment(strategy):
    print(f"\n--- Starting Experiment: {strategy.upper()} ---")
    model = LayeredDepthModel(strategy=strategy).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    criterion = SILogLoss()
    
    # Training Loop (100 steps for verification)
    model.train()
    step = 0
    for sample in tqdm(train_dataset, total=1000, desc="Training"):
        # print(sample.keys()) 
        img = transform_img(sample['image.png'].convert("RGB")).unsqueeze(0).to(device)
        # depth_0 is front, depth_1 is back in Synthetic data
        d0 = transform_depth(sample['depth_1.png']).unsqueeze(0).to(device)
        d1 = transform_depth(sample['depth_2.png']).unsqueeze(0).to(device)
        
        optimizer.zero_grad()
        # Train Layer 1
        p0 = model(img, target_layer=1)
        loss0 = criterion(p0, d0)
        # Train Layer 2
        p1 = model(img, target_layer=2)
        loss1 = criterion(p1, d1)
        
        (loss0 + loss1).backward()
        optimizer.step()
        
        step += 1
        if step >= 1000: break

    acc = evaluate_model(model, val_dataset,len(val_dataset))
    print(f"🎯 Final Real-World Accuracy for {strategy}: {acc:.2f}%")
    return acc

# Setup Transforms
transform_img = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
transform_depth = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# RUN ALL THREE
results = {}
for s in ["multi_head", "recurrent", "concat"]:
    results[s] = run_experiment(s)

print("\n--- FINAL COMPARISON ---")
for s, acc in results.items():
    print(f"{s}: {acc:.2f}%")   


--- Starting Experiment: MULTI_HEAD ---


Using cache found in /home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main
Training: 100%|█████████▉| 999/1000 [08:25<00:00,  1.98it/s]  


🎯 Final Real-World Accuracy for multi_head: 50.66%

--- Starting Experiment: RECURRENT ---


Using cache found in /home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main
Training: 100%|█████████▉| 999/1000 [07:44<00:00,  2.15it/s]  


🎯 Final Real-World Accuracy for recurrent: 50.14%

--- Starting Experiment: CONCAT ---


Using cache found in /home/yash_gupta/.cache/torch/hub/facebookresearch_dinov2_main
Training: 100%|█████████▉| 999/1000 [06:32<00:00,  2.54it/s]  


🎯 Final Real-World Accuracy for concat: 51.10%

--- FINAL COMPARISON ---
multi_head: 50.66%
recurrent: 50.14%
concat: 51.10%
